In [1]:
import ast
import tomllib
from pathlib import Path

from rich import print

In [2]:
toml_config_dir = Path(".")
toml_file = toml_config_dir / "adder.toml"
with open(toml_file, 'rb') as f:
    cfg = tomllib.load(f)

print(cfg)

{
    'name': 'adder',
    'tag': 'dsp:adder',
    'module': 'adder',
    'dsp_block': {
        'hdl_library': 'common_components',
        'sources': [{'path': 'adder/*.v', 'file_type': 'verilog'}],
        'ports': [
            {'name': 'clk', 'dir': 'in', 'width': 1, 'parent_port': True, 'clock': True},
            {'name': 'in0', 'dir': 'in', 'width': 32, 'bus_type': 'axis'},
            {'name': 'in1', 'dir': 'in', 'width': 32, 'bus_type': 'axis'},
            {'name': 'out0', 'dir': 'out', 'width': 32, 'bus_type': 'axis'}
        ]
    }
}

In [8]:
"""
DSP Block generator 
"""

import ast

def gen_imports():
    imports = [
        ast.Import(names=[ast.alias('sys')]),
        ast.ImportFrom(
            module='verilog',
            names=[ast.alias(name='VerilogModule', asname=None)],
            level=0
        ),
        ast.ImportFrom(
            module='.dsp_block',
            names=[ast.alias(name='DSPBlock', asname=None)],
            level=0
        )
    ]
    return imports


def generate_default_method(method_name):
    """Empty method body (just pass)"""
    method_body = [
        ast.Expr(ast.Constant(value=f"{method_name.replace('_', ' ').title()} method")),
        ast.Pass()
    ]
    
    method = ast.FunctionDef(
        name=method_name,
        args=ast.arguments(
            posonlyargs=[],
            args=[ast.arg(arg='self')],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=method_body,
        decorator_list=[],
        lineno=1,  # Add default lineno
        end_lineno=1  # Add end_lineno for Python 3.9+
    )
    return method

def generate_class(class_methods: dict, class_name: str, parent_class: str = "DSPBlock") -> str:
    """Generates a Python class with specified structure using AST nodes"""
    # print(class_methods)
    
    methods = []
    for method_name in ['initialize', 'modify_top']:
        if method_name in class_methods and isinstance(class_methods[method_name], ast.FunctionDef):
            method = class_methods[method_name]
        else:
            method = generate_default_method(method_name)
        methods.append(method)

    bases = [ast.Name(id=parent_class, ctx=ast.Load())] if parent_class else []

    class_def = ast.ClassDef(
        name=class_name,
        bases=bases,
        keywords=[],
        body=methods,
        decorator_list=[],
        lineno=1,  # Add lineno for class definition
        end_lineno=1
    )

    module = ast.Module(
        body=[
            *gen_imports(),
            class_def
        ],
        type_ignores=[]
    )
    ast.fix_missing_locations(module)  # Critical fix for location attributes
    return ast.unparse(module)


In [9]:
"""generate the initialize method"""
def generate_self_dot_func(attr, args):
    dot_call = ast.Call(
        func=ast.Attribute(
            value=ast.Name(id='self', ctx=ast.Load()),  # self
            attr=attr, # .<attr>
            ctx=ast.Load()
        ),
        args=args, # arguments to .<attr>( args )
    )
    return ast.Expr(dot_call)


def generate_initialize(hdl_sources):
    """Create the AST subtree for the initialize method of a DSPBlock.
   the primary job of this method is to register all  HDL files required to build this module.
    """
    method_name = "initialize"

    # generate instructions to add HDL sources
    add_source_instructions = []
    for source in hdl_sources:
        path = source['path']
        add_source_expr = generate_self_dot_func(
            attr='add_source', 
            args=[ast.Constant(value=path)]
        )
        add_source_instructions.append(add_source_expr)

    # assemble body
    method_body = [
        ast.Expr(ast.Constant(value=f"{method_name.replace('_', ' ').title()} method")),
        *add_source_instructions
    ]

    # generate method
    method = ast.FunctionDef(
        name=method_name,
        args=ast.arguments(
            args=[ast.arg(arg='self')],
        ),
        body=method_body,
        lineno=1,  # add default lineno
        end_lineno=1  # add end_lineno for py 3.9+
    )
    return method
    

def generate_modify_top(ports):
    method_name = "modify_top"

    # generate instruction list
    add_port_instructions = []
    for port in ports:
        path = source['path']
        add_port_expr = generate_self_dot_func(
            attr='add_port', 
            args=[
                ast.Constant(value=path)
            ]
        )
        add_port_instructions.append(add_port_expr)

    # assemble body
    method_body = [
        ast.Expr(ast.Constant(value=f"{method_name.replace('_', ' ').title()} method")),
        *add_port_instructions
    ]

    # generate method
    method = ast.FunctionDef(
        name=method_name,
        args=ast.arguments(
            args=[ast.arg(arg='self')],
        ),
        body=method_body,
        lineno=1,  # add default lineno
        end_lineno=1  # add end_lineno for py 3.9+
    )
    return method

In [10]:
# """
# assemble the final DSP module
# """

# def generate_dsp_block(cfg):
#     initialize_method = generate_initialize(cfg['dsp_block']['sources'])
#     modify_top_method = generate_modify_top(cfg['dsp_block']['ports'])
    

#     class_methods = {
#         "initialize": initialize_method,
#         "modify_top": modify_top_method
#     }    
    
#     # Generate the final module
#     generated_code = generate_class(
#         class_methods=class_methods,
#         class_name=cfg['name']
#     )
    
#     print(generated_code)
#     return generated_code

# dsp_block_code = generate_dsp_block(cfg)

In [18]:
import ast
from typing import List, Dict

def generate_verilog_instance(class_name: str, cfg: Dict) -> List[ast.stmt]:
    """Generate Verilog module instantiation code structure"""
    module_name = cfg.get('module', class_name)
    
    return [
        # Create module instance: inst = top.get_instance(entity=module, name=self.fullname)
        ast.Assign(
            targets=[ast.Name(id='inst', ctx=ast.Store())],
            value=ast.Call(
                func=ast.Attribute(
                    value=ast.Name(id='top', ctx=ast.Load()),
                    attr='get_instance',
                    ctx=ast.Load()
                ),
                args=[
                    ast.Constant(value=module_name),
                    ast.Constant(value=f'self.fullname')
                ],
                keywords=[
                    ast.keyword(
                        arg='name',
                        value=ast.Attribute(
                            value=ast.Name(id='self', ctx=ast.Load()),
                            attr='fullname',
                            ctx=ast.Load()
                        )
                    )
                ]
            )
        )
    ]


def generate_port_additions(ports: List[Dict]) -> List[ast.Expr]:
    """Generate AST nodes for adding ports to Verilog instance"""
    port_calls = []
    for port in ports:
        args = [
            
            ast.Constant(value=port['name']),
            ast.BinOp(
                left=ast.Attribute(
                    value=ast.Name(id='self', ctx=ast.Load()),
                    attr='fullname',
                    ctx=ast.Load()
                ),
                op=ast.Add(),
                right=ast.Constant(value=f"_{port['name']}")
            )
            if not port.get('parent_port') 
            else ast.Constant(value=port['name'])
        ]
        
        keywords = [
            ast.keyword(arg='dir', value=ast.Constant(value=port['dir'])),
            ast.keyword(arg='width', value=ast.Constant(value=port['width']))
        ]
        
        if 'bus_type' in port:
            keywords.append(
                ast.keyword(arg='bus_type', value=ast.Constant(value=port['bus_type']))
            )
        
        port_calls.append(
            ast.Expr(
                value=ast.Call(
                    func=ast.Attribute(
                        value=ast.Name(id='inst', ctx=ast.Load()),
                        attr='add_port',
                        ctx=ast.Load()
                    ),
                    args=args,
                    keywords=keywords
                )
            )
        )
    return port_calls

def generate_modify_top(cfg: Dict) -> ast.FunctionDef:
    """Generate complete modify_top method AST"""
    body = [
        # self._populate_parent_ports(top)
        ast.Expr(
            value=ast.Call(
                func=ast.Attribute(
                    value=ast.Name(id='self', ctx=ast.Load()),
                    attr='_populate_parent_ports',
                    ctx=ast.Load()
                ),
                args=[ast.Name(id='top', ctx=ast.Load())],
                keywords=[]
            )
        ),
        *generate_verilog_instance(cfg['name'], cfg),
        *generate_port_additions(cfg['dsp_block']['ports'])
    ]
    
    return ast.FunctionDef(
        name='modify_top',
        args=ast.arguments(
            posonlyargs=[],
            args=[ast.arg(arg='self'), ast.arg(arg='top')],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=body,
        decorator_list=[],
        lineno=1,
        end_lineno=1
    )

def generate_parameter_additions(parameters: List[Dict]) -> List[ast.Expr]:
    """Generate parameter assignment statements"""
    param_calls = []
    for param in parameters:
        param_calls.append(
            ast.Expr(
                value=ast.Call(
                    func=ast.Attribute(
                        value=ast.Name(id='inst', ctx=ast.Load()),
                        attr='add_parameter',
                        ctx=ast.Load()
                    ),
                    args=[
                        ast.Constant(value=param['name']),
                        ast.Constant(value=param['value'])
                    ],
                    keywords=[]
                )
            )
        )
    return param_calls


def generate_dsp_block(cfg: Dict) -> str:
    """Full DSP block generation with parameter handling"""
    # validate_dsp_config(cfg)
    
    class_methods = {
        "initialize": generate_initialize(cfg['dsp_block']['sources']),
        "modify_top": generate_modify_top(cfg)
    }
    
    if 'parameters' in cfg['dsp_block']:
        param_statements = generate_parameter_additions(cfg['dsp_block']['parameters'])
        class_methods['modify_top'].body[2:2] = param_statements  # Insert after instance creation

    generated_code = generate_class(
        class_methods=class_methods,
        class_name=cfg['name'],
        parent_class="DSPBlock"
    )
    
    return generated_code

def validate_dsp_config(cfg: Dict):
    required_fields = {'name', 'tag', 'dsp_block.sources', 'dsp_block.ports'}
    if not required_fields.issubset(cfg.keys()):
        raise ValueError("Missing required configuration fields")
    
    for port in cfg['dsp_block']['ports']:
        if port['dir'] not in {'in', 'out', 'inout'}:
            raise ValueError(f"Invalid direction {port['dir']} for port {port['name']}")


In [25]:
toml_config_dir = Path(".")
toml_file = toml_config_dir / "adder.toml"
with open(toml_file, 'rb') as f:
    cfg = tomllib.load(f)

print(cfg)

{
    'name': 'adder',
    'tag': 'dsp:adder',
    'module': 'adder',
    'dsp_block': {
        'hdl_library': 'common_components',
        'sources': [{'path': 'adder/*.v', 'file_type': 'verilog'}],
        'ports': [
            {'name': 'clk', 'dir': 'in', 'width': 1, 'parent_port': True, 'clock': True},
            {'name': 'in0', 'dir': 'in', 'width': 32, 'bus_type': 'axis'},
            {'name': 'in1', 'dir': 'in', 'width': 32, 'bus_type': 'axis'},
            {'name': 'out0', 'dir': 'out', 'width': 32, 'bus_type': 'axis'}
        ]
    }
}

In [26]:
print(generate_dsp_block(cfg))

import sys
from verilog import VerilogModule
from .dsp_block import DSPBlock

class adder(DSPBlock):

    def initialize(self):
        """Initialize method"""
        self.add_source('adder/*.v')

    def modify_top(self, top):
        self._populate_parent_ports(top)
        inst = top.get_instance('adder', 'self.fullname', name=self.fullname)
        inst.add_port('clk', 'clk', dir='in', width=1)
        inst.add_port('in0', self.fullname + '_in0', dir='in', width=32, bus_type='axis')
        inst.add_port('in1', self.fullname + '_in1', dir='in', width=32, bus_type='axis')
        inst.add_port('out0', self.fullname + '_out0', dir='out', width=32, bus_type='axis')